# Agents

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
	index=index,
	llm_client=openai_client,
	instructions=instructions,
)

In [4]:
search_tool = {
	"type": "function",
	"function": {
		"name": "search",
		"description": "Search the FAQ database for entries matching the given query.",
		"parameters": {
			"type": "object",
			"properties": {
				"query": {
					"type": "string",
					"description": "Search query text to look up in the course FAQ."
				}
			},
			"required": ["query"],
			"additionalProperties": False
		}
	}
}

In [5]:
def search(query):
	boost_dict = {"question": 3.0, "section": 0.5}
	filter_dict = {"course": "llm-zoomcamp"}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict,
		filter_dict=filter_dict
	)

In [6]:
messages = [
  {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [7]:
response = openai_client.chat.completions.create(
	model="gpt-4o-mini",
	messages=messages,
	user="llm-zoomcamp",
	stream=False,
	tools=[search_tool]
)

In [8]:
import json

In [16]:
if response.choices[0].finish_reason == "tool_calls":
	function_call = response.choices[0].message.tool_calls[0].function
  
	if function_call.name == "search":
		args = json.loads(function_call.arguments)
		results = search(**args)
		result_json = json.dumps(results, indent=2)